# 사진 조작·AI 생성 의심 신호 검사 (폴더 단위)

`D:\data` 폴더 안의 사진(및 PDF·Word 등 문서 속 이미지)을 모두 검사해 **조작 또는 AI 생성이 의심되는 신호**를 찾습니다.

**사용법**: 위에서부터 셀을 순서대로 실행하세요 (메뉴 `Run > Run All Cells`). 설정은 **2번 셀**만 바꾸면 됩니다.

| 검사 | 찾는 것 |
|---|---|
| 메타데이터 | AI 생성기 이름·프롬프트, Photoshop 등 편집 프로그램, 카메라 정보 |
| C2PA | 콘텐츠 자격증명에 기록된 AI 생성·편집 이력 |
| ELA | 나중에 붙여 넣거나 고친 영역 (JPEG에서 정확) |
| 노이즈 | 합성된 영역, 지우기·뭉개기·AI 덧칠 |
| 복제-이동 | 같은 사진 안에서 복사·붙여넣기한 부분 |
| AI 판별 API | (선택) 외부 모델의 AI 생성 확률 |

> ⚠️ 이 검사는 진위를 **확정하지 않습니다**. `의심`/`주의`는 사람이 표시된 영역을 직접 확인하라는 뜻이고, `정상`은 "이 검사로는 흔적을 못 찾았다"는 뜻입니다.

## 1. 패키지 설치 (처음 한 번만)

In [ ]:
%pip install -q pillow numpy opencv-python-headless pymupdf c2pa-python pandas matplotlib
# 아이폰 HEIC 사진이 있다면 아래 줄의 주석을 풀고 실행
# %pip install -q pillow-heif

## 2. 설정

In [ ]:
from pathlib import Path
import os

DATA_DIR = Path(r"D:\data")   # 검사할 폴더
RECURSIVE = True               # 하위 폴더까지 검사
INCLUDE_DOCS = True            # PDF/DOCX/PPTX 안의 이미지도 검사
SHOW_LEVEL = "주의"            # 4번 셀에서 그림까지 보여줄 최소 수준: "의심" / "주의" / "참고" / "정상"

REPORT_PATH = DATA_DIR / "imgcheck_report.html"   # HTML 리포트 저장 위치
CSV_PATH = DATA_DIR / "imgcheck_result.csv"       # 엑셀로 열 수 있는 결과표

# (선택) Sightengine AI 판별 API 키. 비워두면 사진이 외부로 전송되지 않습니다.
SIGHTENGINE_API_USER = ""
SIGHTENGINE_API_SECRET = ""

if SIGHTENGINE_API_USER and SIGHTENGINE_API_SECRET:
    os.environ["SIGHTENGINE_API_USER"] = SIGHTENGINE_API_USER
    os.environ["SIGHTENGINE_API_SECRET"] = SIGHTENGINE_API_SECRET

assert DATA_DIR.exists(), f"폴더가 없습니다: {DATA_DIR}"
print("검사 폴더:", DATA_DIR)

## 3. 검사 코드 정의
아래 셀들은 수정할 필요 없이 실행만 하면 됩니다.

In [ ]:
__version__ = "0.1.0"

from __future__ import annotations

from dataclasses import dataclass, field
from enum import IntEnum

from PIL import Image


class Level(IntEnum):
    """신호 수준. 숫자가 클수록 의심스럽다."""

    OK = 0      # 특이사항 없음
    INFO = 1    # 참고 정보
    WARN = 2    # 주의
    ALERT = 3   # 강한 의심 신호

    @property
    def label(self) -> str:
        return {0: "정상", 1: "참고", 2: "주의", 3: "의심"}[self.value]


@dataclass
class Finding:
    level: Level
    message: str


@dataclass
class CheckResult:
    name: str
    title: str
    findings: list[Finding] = field(default_factory=list)
    visual: Image.Image | None = None   # 리포트에 넣을 시각화 이미지
    data: dict = field(default_factory=dict)
    skipped: str | None = None          # 건너뛴 이유

    @property
    def level(self) -> Level:
        if not self.findings:
            return Level.OK
        return max(f.level for f in self.findings)

    def add(self, level: Level, message: str) -> None:
        self.findings.append(Finding(level, message))


@dataclass
class ImageItem:
    """분석 대상 이미지 하나."""

    name: str            # 표시용 이름 (예: page3_img1.jpeg)
    source: str          # 원본 파일 경로
    data: bytes          # 원본 바이트 (재인코딩하지 않은 그대로)
    ext: str             # jpeg, png ...
    location: str = ""   # 문서 내 위치 (예: "3페이지")


@dataclass
class ImageReport:
    item: ImageItem
    width: int = 0
    height: int = 0
    results: list[CheckResult] = field(default_factory=list)
    error: str | None = None

    @property
    def level(self) -> Level:
        if not self.results:
            return Level.OK
        return max(r.level for r in self.results)

    @property
    def verdict(self) -> str:
        return {
            Level.OK: "특이사항 없음",
            Level.INFO: "특이사항 없음 (참고 정보 있음)",
            Level.WARN: "주의 - 사람이 확인 필요",
            Level.ALERT: "조작/AI 생성 의심 신호 있음",
        }[self.level]

In [ ]:
from __future__ import annotations

import cv2
import numpy as np
from PIL import Image

MAX_SIDE = 3000  # 너무 큰 이미지는 분석 속도를 위해 줄인다


def to_rgb(img: Image.Image, max_side: int = MAX_SIDE) -> Image.Image:
    img = img.convert("RGB")
    if max(img.size) > max_side:
        img = img.copy()
        img.thumbnail((max_side, max_side), Image.LANCZOS)
    return img


def block_map(values: np.ndarray, block: int) -> np.ndarray:
    """2차원 배열을 block x block 칸 단위 평균으로 줄인다."""
    h, w = values.shape
    h2, w2 = h // block, w // block
    v = values[: h2 * block, : w2 * block]
    return v.reshape(h2, block, w2, block).mean(axis=(1, 3))


def outlier_ratio(blocks: np.ndarray, mask: np.ndarray | None = None, z: float = 4.0) -> tuple[float, np.ndarray]:
    """중앙값/MAD 기준으로 튀는 칸의 비율과 위치를 구한다."""
    vals = blocks[mask] if mask is not None else blocks.ravel()
    if vals.size < 16:
        return 0.0, np.zeros_like(blocks, dtype=bool)
    med = np.median(vals)
    mad = np.median(np.abs(vals - med)) * 1.4826 + 1e-6
    out = (blocks - med) / mad > z
    if mask is not None:
        out &= mask
    return float(out.sum()) / vals.size, out


def heatmap_overlay(base: Image.Image, values: np.ndarray, outliers: np.ndarray | None = None) -> Image.Image:
    """값 배열을 컬러 히트맵으로 만들어 원본 위에 겹친다. 튀는 칸은 빨간 테두리로 표시."""
    base_np = np.array(base.convert("RGB"))
    h, w = base_np.shape[:2]
    v = values.astype(np.float32)
    hi = np.percentile(v, 99.5) or 1.0
    v = np.clip(v / hi, 0, 1)
    heat = cv2.applyColorMap((v * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heat = cv2.resize(heat, (w, h), interpolation=cv2.INTER_NEAREST)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    out = cv2.addWeighted(base_np, 0.45, heat, 0.55, 0)
    if outliers is not None and outliers.any():
        bh, bw = h / outliers.shape[0], w / outliers.shape[1]
        for y, x in zip(*np.nonzero(outliers)):
            cv2.rectangle(out, (int(x * bw), int(y * bh)), (int((x + 1) * bw), int((y + 1) * bh)), (255, 0, 0), 2)
    return Image.fromarray(out)


def largest_cluster(outliers: np.ndarray) -> int:
    """튀는 칸 중 서로 붙어 있는 가장 큰 덩어리의 칸 수."""
    if not outliers.any():
        return 0
    n, labels, stats, _ = cv2.connectedComponentsWithStats(outliers.astype(np.uint8), connectivity=8)
    return int(stats[1:, cv2.CC_STAT_AREA].max()) if n > 1 else 0

In [ ]:
"""입력 파일(PDF, Office 문서, 이미지)에서 이미지를 원본 바이트 그대로 꺼낸다."""

from __future__ import annotations

import io
import zipfile
from pathlib import Path

from PIL import Image


IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff", ".bmp", ".gif", ".heic", ".avif"}
OFFICE_EXTS = {".docx", ".pptx", ".xlsx", ".hwpx", ".odt", ".odp"}
MIN_SIDE = 64  # 아이콘/장식 등 너무 작은 이미지는 건너뜀


def _too_small(data: bytes) -> bool:
    try:
        with Image.open(io.BytesIO(data)) as im:
            return min(im.size) < MIN_SIDE
    except Exception:
        return True  # 열 수 없는 형식(EMF/WMF 등)은 제외


def extract_images(path: str | Path) -> list[ImageItem]:
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        return _from_pdf(path)
    if suffix in OFFICE_EXTS:
        return _from_zip(path)
    if suffix in IMAGE_EXTS:
        ext = suffix.lstrip(".").replace("jpg", "jpeg")
        return [ImageItem(name=path.name, source=str(path), data=path.read_bytes(), ext=ext)]
    raise ValueError(f"지원하지 않는 파일 형식입니다: {path.name}")


def _from_pdf(path: Path) -> list[ImageItem]:
    import pymupdf

    items: list[ImageItem] = []
    seen: set[int] = set()
    with pymupdf.open(path) as doc:
        for page_no, page in enumerate(doc, start=1):
            for idx, img in enumerate(page.get_images(full=True), start=1):
                xref = img[0]
                if xref in seen:
                    continue
                seen.add(xref)
                info = doc.extract_image(xref)
                if not info or _too_small(info["image"]):
                    continue
                ext = info.get("ext", "png").replace("jpg", "jpeg")
                items.append(ImageItem(
                    name=f"p{page_no}_img{idx}.{ext}",
                    source=str(path),
                    data=info["image"],
                    ext=ext,
                    location=f"{page_no}페이지",
                ))
    return items


def _from_zip(path: Path) -> list[ImageItem]:
    items: list[ImageItem] = []
    with zipfile.ZipFile(path) as zf:
        for name in sorted(zf.namelist()):
            ext = Path(name).suffix.lower()
            if ext not in IMAGE_EXTS:
                continue
            data = zf.read(name)
            if _too_small(data):
                continue
            items.append(ImageItem(
                name=Path(name).name,
                source=str(path),
                data=data,
                ext=ext.lstrip(".").replace("jpg", "jpeg"),
                location=name,
            ))
    return items

In [ ]:
"""EXIF / XMP / PNG 텍스트 등 메타데이터에서 AI 생성·편집 흔적을 찾는다."""

from __future__ import annotations

import io
import re

from PIL import ExifTags, Image


AI_KEYWORDS = [
    "midjourney", "stable diffusion", "stablediffusion", "sdxl", "dall-e", "dall·e", "dalle",
    "openai", "chatgpt", "gpt-4o", "sora", "firefly", "imagen", "gemini", "nano banana",
    "comfyui", "automatic1111", "invokeai", "novelai", "leonardo.ai", "ideogram",
    "black forest labs", "bing image creator", "microsoft designer", "grok",
    "trainedalgorithmicmedia", "compositewithtrainedalgorithmicmedia",
    "algorithmicmedia", "generative fill", "made with ai", "ai generated",
]

EDIT_KEYWORDS = [
    "photoshop", "lightroom", "gimp", "affinity", "pixelmator", "snapseed", "picsart",
    "canva", "facetune", "meitu", "photoscape", "paint.net", "corel", "capture one",
    "luminar", "fotor", "remini", "lensa",
]

# PNG 텍스트 청크에 생성 정보를 남기는 도구들의 키
AI_PNG_KEYS = {"parameters", "prompt", "workflow", "sd-metadata", "dream", "invokeai_metadata"}

# 이미지 생성 모델에서 흔한 해상도 (약한 신호)
AI_TYPICAL_SIZES = {
    (512, 512), (768, 768), (1024, 1024), (2048, 2048),
    (1024, 1792), (1792, 1024), (1024, 1536), (1536, 1024),
    (832, 1216), (1216, 832), (896, 1152), (1152, 896), (768, 1344), (1344, 768),
    (640, 1536), (1536, 640), (1456, 816), (816, 1456), (1344, 896), (896, 1344),
}


def _find(keywords: list[str], text: str) -> list[str]:
    t = text.lower()
    hits = []
    for k in keywords:
        # 짧은 단어(flux, grok 등)는 단어 경계로만 매칭해 오탐을 줄인다
        if re.search(rf"(?<![a-z]){re.escape(k)}(?![a-z])", t):
            hits.append(k)
    return hits


def check_metadata(item: ImageItem, img: Image.Image) -> CheckResult:
    r = CheckResult("metadata", "메타데이터 (EXIF/XMP)")

    with Image.open(io.BytesIO(item.data)) as raw:
        exif = raw.getexif()
        info = dict(raw.info)

    tags: dict[str, str] = {}
    for tag_id, value in exif.items():
        tags[ExifTags.TAGS.get(tag_id, str(tag_id))] = str(value)
    try:
        for tag_id, value in exif.get_ifd(ExifTags.IFD.Exif).items():
            tags[ExifTags.TAGS.get(tag_id, str(tag_id))] = str(value)
    except Exception:
        pass

    xmp = info.get("xmp") or info.get("XML:com.adobe.xmp") or b""
    if isinstance(xmp, bytes):
        xmp = xmp.decode("utf-8", "ignore")

    png_text = {k: str(v) for k, v in info.items()
                if isinstance(v, str) and k not in ("xmp", "XML:com.adobe.xmp")}

    keep = ["Make", "Model", "Software", "DateTime", "DateTimeOriginal",
            "DateTimeDigitized", "Artist", "ImageDescription", "LensModel"]
    r.data = {
        "exif": {k: tags[k][:200] for k in keep if k in tags},
        "has_xmp": bool(xmp),
        "png_text_keys": sorted(png_text.keys()),
    }

    all_text = " ".join(tags.values()) + " " + xmp + " " + " ".join(
        f"{k} {v}" for k, v in png_text.items())

    ai_hits = _find(AI_KEYWORDS, all_text)
    ai_png = AI_PNG_KEYS & {k.lower() for k in png_text}
    if ai_hits:
        r.add(Level.ALERT, f"AI 생성 도구 흔적이 메타데이터에 있습니다: {', '.join(ai_hits)}")
    if ai_png:
        r.add(Level.ALERT, f"AI 생성 프로그램이 남기는 PNG 정보(프롬프트 등)가 있습니다: {', '.join(sorted(ai_png))}")

    edit_hits = _find(EDIT_KEYWORDS, all_text)
    if edit_hits:
        r.add(Level.WARN, f"편집 프로그램 흔적이 있습니다: {', '.join(edit_hits)} "
                          "(밝기 보정 같은 단순 편집일 수도 있음)")

    if "Make" in tags or "Model" in tags:
        cam = f"{tags.get('Make', '').strip()} {tags.get('Model', '').strip()}".strip()
        r.add(Level.OK, f"카메라 정보 있음: {cam}")
    else:
        r.add(Level.INFO, "카메라 정보가 없습니다. 메타데이터는 문서에 넣거나 메신저로 보낼 때 "
                          "흔히 지워지므로 이것만으로 조작을 뜻하지는 않습니다.")

    orig, mod = tags.get("DateTimeOriginal"), tags.get("DateTime")
    if orig and mod and orig != mod:
        r.add(Level.INFO, f"촬영 시각({orig})과 수정 시각({mod})이 다릅니다.")

    if img.size in AI_TYPICAL_SIZES and "Make" not in tags:
        r.add(Level.INFO, f"해상도 {img.size[0]}x{img.size[1]}는 AI 이미지 생성기에서 흔한 크기입니다 (약한 신호).")

    return r

In [ ]:
"""C2PA(콘텐츠 자격증명) 매니페스트를 읽어 AI 생성/편집 이력을 확인한다."""

from __future__ import annotations

import io
import json

from PIL import Image


MIME = {
    "jpeg": "image/jpeg", "png": "image/png", "webp": "image/webp",
    "tif": "image/tiff", "tiff": "image/tiff", "heic": "image/heic", "avif": "image/avif",
}


def check_c2pa(item: ImageItem, img: Image.Image) -> CheckResult:
    r = CheckResult("c2pa", "C2PA 출처 정보 (콘텐츠 자격증명)")
    try:
        import c2pa
    except ImportError:
        r.skipped = "c2pa-python 이 설치되지 않음"
        return r

    mime = MIME.get(item.ext)
    if not mime:
        r.skipped = f"{item.ext} 형식은 C2PA 확인을 지원하지 않음"
        return r

    try:
        reader = c2pa.Reader(mime, io.BytesIO(item.data))
        store = json.loads(reader.json())
        state = reader.get_validation_state()
    except Exception:
        r.add(Level.OK, "C2PA 정보가 없습니다 (대부분의 이미지는 C2PA 정보가 없음).")
        return r

    manifests = store.get("manifests", {})
    active = manifests.get(store.get("active_manifest", ""), {})
    generator = active.get("claim_generator") or ", ".join(
        g.get("name", "") for g in active.get("claim_generator_info", []))
    r.data = {"validation_state": str(state), "claim_generator": generator}

    text = json.dumps(store).lower()
    if "trainedalgorithmicmedia" in text:
        r.add(Level.ALERT, "C2PA 기록상 AI로 생성되었거나 AI 생성 요소가 포함된 이미지입니다.")
    actions = set()
    for m in manifests.values():
        for a in m.get("assertions", []):
            if a.get("label", "").startswith("c2pa.actions"):
                for act in a.get("data", {}).get("actions", []):
                    actions.add(act.get("action", ""))
    edit_actions = actions - {"c2pa.created", "c2pa.opened", ""}
    if edit_actions:
        r.add(Level.WARN, f"C2PA 편집 이력: {', '.join(sorted(edit_actions))}")
    if generator:
        r.add(Level.INFO, f"생성/서명 프로그램: {generator}")
    if str(state).lower() not in ("valid", "trusted"):
        r.add(Level.WARN, f"C2PA 서명 검증 상태: {state} (서명 이후 변경되었거나 신뢰할 수 없는 서명일 수 있음)")
    return r

In [ ]:
"""ELA(Error Level Analysis, 압축 오류 분석).

JPEG를 같은 품질로 다시 저장하면, 원래 함께 압축된 영역은 차이가 비슷하게 나오고
나중에 붙여 넣거나 고친 영역은 차이가 다르게 나오는 경향이 있다.
"""

from __future__ import annotations

import io

import numpy as np
from PIL import Image, ImageChops


BLOCK = 16
QUALITY = 90


def check_ela(item: ImageItem, img: Image.Image) -> CheckResult:
    r = CheckResult("ela", "ELA 압축 오류 분석 (부분 편집 탐지)")
    rgb = to_rgb(img)

    buf = io.BytesIO()
    rgb.save(buf, "JPEG", quality=QUALITY)
    resaved = Image.open(io.BytesIO(buf.getvalue()))
    diff = np.asarray(ImageChops.difference(rgb, resaved), dtype=np.float32).max(axis=2)

    blocks = block_map(diff, BLOCK)
    # 완전히 평평한 영역(하늘, 흰 배경)은 원래 차이가 작으므로 판단에서 뺀다
    gray = np.asarray(rgb.convert("L"), dtype=np.float32)
    texture = block_map(np.abs(np.diff(gray, axis=1, append=gray[:, -1:])), BLOCK)
    mask = texture > 1.0

    ratio, out = outlier_ratio(blocks, mask, z=5.0)
    cluster = largest_cluster(out)
    r.data = {"outlier_ratio": round(ratio, 4), "largest_cluster_blocks": cluster,
              "mean_error": round(float(diff.mean()), 3)}

    # 사람이 보기 쉽게 차이를 증폭한 ELA 이미지 대신, 원본 위 히트맵을 보여준다
    r.visual = heatmap_overlay(rgb, blocks, out)

    if item.ext != "jpeg":
        r.add(Level.INFO, f"{item.ext.upper()} 이미지라 ELA 신뢰도가 낮습니다 (JPEG에서 가장 잘 동작).")

    # 튀는 칸이 이미지 일부에만 한 덩어리로 몰려 있을 때만 경고한다
    n_valid = max(int(mask.sum()), 1)
    if cluster >= max(12, n_valid * 0.015) and ratio < 0.1:
        r.add(Level.WARN, f"압축 오류가 주변과 다른 영역이 한 덩어리로 있습니다 ({cluster}칸). "
                          "히트맵의 빨간 테두리 영역을 눈으로 확인하세요. 글자, 경계선, 강한 대비 영역은 "
                          "원래도 밝게 나올 수 있습니다.")
    elif ratio >= 0.1:
        r.add(Level.INFO, "이미지 전체에 압축 오류가 고르게 높습니다. 여러 번 재압축되었거나 "
                          "텍스처가 많은 이미지로 보입니다.")
    else:
        r.add(Level.OK, "압축 오류가 이미지 전체에서 비교적 고르게 나타납니다.")
    return r

In [ ]:
"""노이즈 일관성 분석.

한 장의 사진은 보통 센서 노이즈가 전체적으로 비슷하다. 다른 사진을 합성하거나
일부를 뭉개고(스탬프/블러) 다시 그리면 그 영역의 노이즈 수준이 달라진다.
"""

from __future__ import annotations

import cv2
import numpy as np
from PIL import Image


BLOCK = 32


def check_noise(item: ImageItem, img: Image.Image) -> CheckResult:
    r = CheckResult("noise", "노이즈 일관성 (합성/덧칠 탐지)")
    rgb = to_rgb(img)
    gray = np.asarray(rgb.convert("L"), dtype=np.float32)
    if min(gray.shape) < BLOCK * 4:
        r.skipped = "이미지가 너무 작음"
        return r

    residual = gray - cv2.medianBlur(gray.astype(np.uint8), 3).astype(np.float32)
    noise = np.sqrt(block_map(residual ** 2, BLOCK))

    # 노이즈는 무늬가 많은 곳에서 크게 나오므로, 무늬가 적은(평탄한) 칸끼리만 비교한다
    bright = block_map(gray, BLOCK)
    smooth = cv2.GaussianBlur(gray, (0, 0), 2.0)
    gx, gy = np.gradient(smooth)
    structure = block_map(np.hypot(gx, gy), BLOCK)
    valid = (bright > 8) & (bright < 247)
    if valid.sum() < 16:
        r.skipped = "분석할 영역이 부족함"
        return r
    mask = valid & (structure <= np.percentile(structure[valid], 50))
    vals = noise[mask]
    if vals.size < 16:
        r.skipped = "분석할 영역이 부족함"
        return r

    med = float(np.median(vals))
    # 주변보다 노이즈가 비정상적으로 "낮은" 영역(뭉개기/AI 덧칠)과 "높은" 영역을 모두 본다
    low = mask & (noise < med * 0.4)
    high = mask & (noise > med * 2.5)
    low_c, high_c = largest_cluster(low), largest_cluster(high)
    r.data = {"median_noise": round(med, 3), "low_cluster": low_c, "high_cluster": high_c,
              "compared_blocks": int(mask.sum())}
    r.visual = heatmap_overlay(rgb, noise, low | high)

    if med < 0.6:
        r.add(Level.INFO, "이미지 전체 노이즈가 매우 낮습니다. 그래픽/스크린샷이거나, 강한 보정 또는 "
                          "AI 생성 이미지일 수 있습니다 (약한 신호).")

    total = int(mask.sum())
    if low_c >= max(10, total * 0.025) and low_c < total * 0.5:
        r.add(Level.WARN, f"주변보다 노이즈가 훨씬 적은 영역이 있습니다 ({low_c}칸). "
                          "지우기/복제 도장/AI 덧칠 등이 의심되지만, 초점이 나간 배경도 이렇게 보일 수 있습니다.")
    if high_c >= max(15, total * 0.08) and high_c < total * 0.5:
        r.add(Level.WARN, f"주변보다 노이즈가 훨씬 많은 영역이 있습니다 ({high_c}칸). "
                          "다른 이미지를 합성한 흔적일 수 있으나 복잡한 무늬도 이렇게 보일 수 있습니다.")
    if not r.findings:
        r.add(Level.OK, "노이즈 수준이 이미지 전체에서 비교적 일정합니다.")
    return r

In [ ]:
"""복제-이동(Copy-Move) 탐지.

같은 이미지 안에서 한 부분을 복사해 다른 곳에 붙이면(물체 복제, 지우기용 도장 등)
똑같은 특징점 무리가 "같은 방향·같은 거리"만큼 떨어져 나타난다.
"""

from __future__ import annotations

from collections import defaultdict

import cv2
import numpy as np
from PIL import Image


MIN_DIST = 40      # 너무 가까운 특징점끼리는 같은 물체로 보고 무시
BIN = 8            # 이동 벡터를 묶는 단위(픽셀)
WARN_AT = 10
ALERT_AT = 25


def check_copymove(item: ImageItem, img: Image.Image) -> CheckResult:
    r = CheckResult("copymove", "복제-이동 탐지 (같은 이미지 내 복사·붙여넣기)")
    rgb = to_rgb(img, max_side=1600)
    gray = cv2.cvtColor(np.array(rgb), cv2.COLOR_RGB2GRAY)

    sift = cv2.SIFT_create(nfeatures=4000)
    kps, desc = sift.detectAndCompute(gray, None)
    if desc is None or len(kps) < 20:
        r.add(Level.OK, "특징점이 적어 복제 영역을 찾지 못했습니다 (단순한 그림/그래픽일 수 있음).")
        return r

    matcher = cv2.BFMatcher(cv2.NORM_L2)
    knn = matcher.knnMatch(desc, desc, k=3)

    groups: dict[tuple[int, int], list[tuple[tuple[float, float], tuple[float, float]]]] = defaultdict(list)
    for m in knn:
        if len(m) < 3:
            continue
        # m[0]은 자기 자신. m[1]이 m[2]보다 확실히 가까울 때만 채택(비율 검사)
        best, second = m[1], m[2]
        if best.distance > 0.6 * second.distance:
            continue
        p1 = np.array(kps[best.queryIdx].pt)
        p2 = np.array(kps[best.trainIdx].pt)
        d = p2 - p1
        if np.hypot(*d) < MIN_DIST:
            continue
        # 방향을 통일해 A→B 와 B→A 를 같은 묶음으로 본다
        if d[0] < 0 or (d[0] == 0 and d[1] < 0):
            d, p1, p2 = -d, p2, p1
        key = (int(round(d[0] / BIN)), int(round(d[1] / BIN)))
        groups[key].append((tuple(p1), tuple(p2)))

    # 서로 다른 출발점만 센다 (같은 매칭이 양방향으로 두 번 들어가는 것 방지)
    clusters = sorted(
        ((k, list({p1: (p1, p2) for p1, p2 in v}.values())) for k, v in groups.items()),
        key=lambda kv: len(kv[1]), reverse=True,
    )
    top = len(clusters[0][1]) if clusters else 0
    r.data = {"keypoints": len(kps), "largest_group": top}

    if top >= WARN_AT:
        vis = np.array(rgb).copy()
        colors = [(255, 0, 0), (255, 160, 0), (0, 200, 255)]
        for (key, pairs), color in zip(clusters[:3], colors):
            if len(pairs) < WARN_AT:
                break
            for p1, p2 in pairs:
                a = tuple(int(v) for v in p1)
                b = tuple(int(v) for v in p2)
                cv2.line(vis, a, b, color, 1, cv2.LINE_AA)
                cv2.circle(vis, a, 4, color, 2)
                cv2.circle(vis, b, 4, color, 2)
        r.visual = Image.fromarray(vis)
        level = Level.ALERT if top >= ALERT_AT else Level.WARN
        r.add(level, f"같은 모양이 일정한 거리만큼 떨어져 반복되는 부분이 있습니다 (일치점 {top}개). "
                     "복사·붙여넣기 흔적일 수 있습니다. 단, 타일·창문·글자처럼 원래 반복되는 무늬도 "
                     "이렇게 나오므로 선으로 연결된 부분을 눈으로 확인하세요.")
    else:
        r.add(Level.OK, "이미지 안에서 복사·붙여넣기된 영역은 발견되지 않았습니다.")
    return r

In [ ]:
"""(선택) 외부 AI 생성 판별 API 연동.

환경변수 SIGHTENGINE_API_USER, SIGHTENGINE_API_SECRET 이 설정되어 있을 때만 동작한다.
설정하지 않으면 이 검사는 건너뛰며, 이미지가 외부로 전송되지 않는다.
"""

from __future__ import annotations

import json
import os
import urllib.request
import uuid

from PIL import Image


URL = "https://api.sightengine.com/1.0/check.json"


def _multipart(fields: dict[str, str], file_bytes: bytes, filename: str) -> tuple[bytes, str]:
    boundary = uuid.uuid4().hex
    parts = []
    for k, v in fields.items():
        parts.append(f'--{boundary}\r\nContent-Disposition: form-data; name="{k}"\r\n\r\n{v}\r\n'.encode())
    parts.append(
        f'--{boundary}\r\nContent-Disposition: form-data; name="media"; filename="{filename}"\r\n'
        f"Content-Type: application/octet-stream\r\n\r\n".encode() + file_bytes + b"\r\n")
    parts.append(f"--{boundary}--\r\n".encode())
    return b"".join(parts), f"multipart/form-data; boundary={boundary}"


def check_ai_api(item: ImageItem, img: Image.Image) -> CheckResult:
    r = CheckResult("ai_api", "AI 생성 판별 (외부 API)")
    user = os.environ.get("SIGHTENGINE_API_USER")
    secret = os.environ.get("SIGHTENGINE_API_SECRET")
    if not (user and secret):
        r.skipped = "API 키 미설정 (README의 'AI 판별 API 연동' 참고)"
        return r

    body, ctype = _multipart({"models": "genai", "api_user": user, "api_secret": secret},
                             item.data, item.name)
    req = urllib.request.Request(URL, data=body, headers={"Content-Type": ctype}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            res = json.load(resp)
    except Exception as e:
        r.skipped = f"API 호출 실패: {e}"
        return r

    if res.get("status") != "success":
        r.skipped = f"API 오류: {res.get('error', res)}"
        return r

    score = float(res.get("type", {}).get("ai_generated", 0.0))
    r.data = {"ai_generated": score}
    pct = f"{score * 100:.0f}%"
    if score >= 0.85:
        r.add(Level.ALERT, f"AI 생성 가능성이 높게 나왔습니다 ({pct}).")
    elif score >= 0.5:
        r.add(Level.WARN, f"AI 생성 가능성이 중간 정도로 나왔습니다 ({pct}).")
    else:
        r.add(Level.OK, f"AI 생성 가능성이 낮게 나왔습니다 ({pct}).")
    r.add(Level.INFO, "판별 모델 결과는 확률이며 오탐·미탐이 있을 수 있습니다.")
    return r

In [ ]:
ALL_CHECKS = [check_metadata, check_c2pa, check_ela, check_noise, check_copymove, check_ai_api]

from __future__ import annotations

import io

from PIL import Image, ImageOps


Image.MAX_IMAGE_PIXELS = 200_000_000

try:  # HEIC/AVIF 지원 (설치되어 있을 때만)
    from pillow_heif import register_heif_opener
    register_heif_opener()
except ImportError:
    pass


def analyze_image(item: ImageItem, checks=None) -> ImageReport:
    rep = ImageReport(item=item)
    try:
        img = Image.open(io.BytesIO(item.data))
        img.load()
        img = ImageOps.exif_transpose(img)
    except Exception as e:
        rep.error = f"이미지를 열 수 없습니다: {e}"
        return rep
    rep.width, rep.height = img.size

    for check in checks or ALL_CHECKS:
        try:
            rep.results.append(check(item, img))
        except Exception as e:  # 한 검사가 실패해도 나머지는 계속
            res = CheckResult(check.__name__, check.__name__)
            res.skipped = f"검사 중 오류: {e}"
            rep.results.append(res)
    return rep

In [ ]:
"""분석 결과를 HTML 한 파일(이미지 포함)과 JSON으로 저장한다."""

from __future__ import annotations

import base64
import html
import io
import json
from datetime import datetime
from pathlib import Path

from PIL import Image


THUMB = 900

CSS = """
:root{--bg:#f6f7f9;--card:#fff;--text:#1d2129;--muted:#6b7280;--line:#e5e7eb;
--ok:#15803d;--info:#2563eb;--warn:#b45309;--alert:#b91c1c}
@media (prefers-color-scheme:dark){:root{--bg:#111318;--card:#1b1e25;--text:#e6e8ec;
--muted:#9aa1ad;--line:#2c313a;--ok:#4ade80;--info:#60a5fa;--warn:#fbbf24;--alert:#f87171}}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--text);
font:15px/1.6 -apple-system,"Apple SD Gothic Neo","Malgun Gothic",sans-serif}
main{max-width:1000px;margin:0 auto;padding:24px 16px}
h1{font-size:22px;margin:0 0 4px}h2{font-size:17px;margin:0}
.muted{color:var(--muted);font-size:13px}
.card{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:16px;margin:16px 0}
.notice{border-left:4px solid var(--warn)}
table{width:100%;border-collapse:collapse;font-size:14px}
td,th{padding:6px 8px;border-bottom:1px solid var(--line);text-align:left;vertical-align:top}
.badge{display:inline-block;padding:1px 8px;border-radius:99px;font-size:12px;font-weight:600;
border:1px solid currentColor;white-space:nowrap}
.l0{color:var(--ok)}.l1{color:var(--info)}.l2{color:var(--warn)}.l3{color:var(--alert)}
img{max-width:100%;height:auto;border-radius:6px;border:1px solid var(--line)}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(260px,1fr));gap:12px}
details{margin-top:10px}summary{cursor:pointer;font-weight:600}
ul{margin:6px 0;padding-left:20px}
pre{white-space:pre-wrap;word-break:break-all;font-size:12px;background:var(--bg);padding:8px;border-radius:6px}
"""


def _img_b64(img: Image.Image) -> str:
    img = img.convert("RGB")
    img.thumbnail((THUMB, THUMB))
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=85)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()


def _badge(level: Level) -> str:
    return f'<span class="badge l{int(level)}">{level.label}</span>'


def _image_section(i: int, rep: ImageReport) -> str:
    it = rep.item
    e = html.escape
    out = [f'<section class="card" id="img{i}">']
    out.append(f"<h2>{i}. {e(it.name)} {_badge(rep.level)}</h2>")
    meta = f"{e(it.location)} · " if it.location else ""
    out.append(f'<div class="muted">{meta}{rep.width}x{rep.height} · {it.ext.upper()} · '
               f"{len(it.data) // 1024}KB</div>")
    if rep.error:
        out.append(f"<p>{e(rep.error)}</p></section>")
        return "".join(out)
    out.append(f"<p><b>판정: {e(rep.verdict)}</b></p>")

    try:
        orig = Image.open(io.BytesIO(it.data))
        out.append(f'<img src="{_img_b64(orig)}" alt="원본">')
    except Exception:
        pass

    for res in rep.results:
        out.append(f"<details {'open' if res.level >= Level.WARN else ''}>")
        status = "건너뜀" if res.skipped else res.level.label
        cls = "l1" if res.skipped else f"l{int(res.level)}"
        out.append(f'<summary>{e(res.title)} <span class="badge {cls}">{status}</span></summary>')
        if res.skipped:
            out.append(f'<p class="muted">{e(res.skipped)}</p>')
        if res.findings:
            out.append("<ul>")
            for f in sorted(res.findings, key=lambda f: -f.level):
                out.append(f"<li>{_badge(f.level)} {e(f.message)}</li>")
            out.append("</ul>")
        if res.visual is not None:
            out.append(f'<img src="{_img_b64(res.visual)}" alt="{e(res.title)} 시각화">')
        if res.data:
            out.append(f"<pre>{e(json.dumps(res.data, ensure_ascii=False, indent=2))}</pre>")
        out.append("</details>")
    out.append("</section>")
    return "".join(out)


def write_html(reports: list[ImageReport], sources: list[str], path: str | Path) -> None:
    e = html.escape
    now = datetime.now().strftime("%Y-%m-%d %H:%M")
    counts = {lv: sum(1 for r in reports if r.level == lv) for lv in Level}

    rows = "".join(
        f'<tr><td><a href="#img{i}">{i}</a></td><td>{e(r.item.name)}</td>'
        f"<td>{e(r.item.location)}</td><td>{_badge(r.level)} {e(r.verdict)}</td></tr>"
        for i, r in enumerate(reports, start=1))

    body = f"""
<h1>이미지 진위 분석 리포트</h1>
<div class="muted">분석 대상: {e(', '.join(Path(s).name for s in sources))} · {now} · imgcheck {__version__}</div>

<div class="card">
<b>요약</b> — 이미지 {len(reports)}개:
{_badge(Level.ALERT)} {counts[Level.ALERT]} &nbsp;
{_badge(Level.WARN)} {counts[Level.WARN]} &nbsp;
{_badge(Level.INFO)} {counts[Level.INFO]} &nbsp;
{_badge(Level.OK)} {counts[Level.OK]}
<table style="margin-top:10px"><tr><th>#</th><th>이미지</th><th>위치</th><th>판정</th></tr>{rows}</table>
</div>

<div class="card notice">
<b>해석 시 주의</b>
<ul>
<li>이 리포트는 조작·AI 생성의 <b>의심 신호</b>를 보여줄 뿐, 진위를 확정하지 않습니다.</li>
<li>'특이사항 없음'은 "이 검사들로는 흔적을 찾지 못했다"는 뜻이지 진본 보증이 아닙니다.</li>
<li>문서(PDF/Word)에 넣은 이미지는 메타데이터가 지워지거나 재압축되는 경우가 많아, 가능하면 원본 파일로 다시 검사하세요.</li>
<li>법적 분쟁 등 중요한 사안은 전문 디지털 포렌식 감정을 권합니다.</li>
</ul>
</div>
{''.join(_image_section(i, r) for i, r in enumerate(reports, start=1))}
"""
    page = (f'<!doctype html><html lang="ko"><head><meta charset="utf-8">'
            f'<meta name="viewport" content="width=device-width,initial-scale=1">'
            f"<title>이미지 진위 분석</title><style>{CSS}</style></head>"
            f"<body><main>{body}</main></body></html>")
    Path(path).write_text(page, encoding="utf-8")


def write_json(reports: list[ImageReport], path: str | Path) -> None:
    data = []
    for r in reports:
        data.append({
            "name": r.item.name, "source": r.item.source, "location": r.item.location,
            "width": r.width, "height": r.height, "level": r.level.name, "verdict": r.verdict,
            "error": r.error,
            "checks": [{
                "name": c.name, "level": c.level.name, "skipped": c.skipped,
                "findings": [{"level": f.level.name, "message": f.message} for f in c.findings],
                "data": c.data,
            } for c in r.results],
        })
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

## 4. 폴더 검사 실행

In [ ]:
import time

pattern = "**/*" if RECURSIVE else "*"
exts = set(IMAGE_EXTS) | ({".pdf"} | OFFICE_EXTS if INCLUDE_DOCS else set())
files = sorted(p for p in DATA_DIR.glob(pattern) if p.is_file() and p.suffix.lower() in exts)
print(f"대상 파일 {len(files)}개")

items = []
for f in files:
    try:
        found = extract_images(f)
    except Exception as e:
        print(f"  [건너뜀] {f.name}: {e}")
        continue
    rel = str(f.relative_to(DATA_DIR))
    for it in found:
        it.name = rel if f.suffix.lower() in IMAGE_EXTS else f"{rel} :: {it.name}"
    items.extend(found)
print(f"검사할 이미지 {len(items)}개\n")

reports = []
t0 = time.time()
for n, it in enumerate(items, start=1):
    rep = analyze_image(it)
    reports.append(rep)
    print(f"[{n}/{len(items)}] {rep.level.label:>2} | {it.name}")
print(f"\n완료: {time.time() - t0:.1f}초")

## 5. 결과 요약표
의심 → 주의 → 참고 → 정상 순으로 정렬됩니다.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)

SHORT = {"metadata": "메타데이터", "c2pa": "C2PA", "ela": "ELA", "noise": "노이즈",
         "copymove": "복제", "ai_api": "AI판별"}

rows = []
for r in reports:
    row = {"판정": r.level.label, "파일": r.item.name, "크기": f"{r.width}x{r.height}"}
    for res in r.results:
        row[SHORT.get(res.name, res.name)] = "건너뜀" if res.skipped else res.level.label
    row["주요 신호"] = " / ".join(f.message for res in r.results for f in res.findings
                                if f.level >= Level.WARN) or r.error or ""
    row["_lv"] = int(r.level)
    rows.append(row)

df = pd.DataFrame(rows).sort_values(["_lv", "파일"], ascending=[False, True]).drop(columns="_lv")
print(df["판정"].value_counts().to_string())
df.reset_index(drop=True)

## 6. 의심 사진 자세히 보기
원본과 함께, 의심 영역을 표시한 그림(빨간 테두리·선)을 보여줍니다.

In [ ]:
import io, platform
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

LEVEL_BY_LABEL = {lv.label: lv for lv in Level}


def show_report(r, only_flagged=True):
    print("=" * 100)
    print(f"[{r.level.label}] {r.item.name}  ({r.width}x{r.height})  → {r.verdict}")
    for res in r.results:
        for f in sorted(res.findings, key=lambda f: -f.level):
            if f.level >= Level.INFO:
                print(f"   - [{f.level.label}] {res.title}: {f.message}")
    panels = [("원본", Image.open(io.BytesIO(r.item.data)))]
    panels += [(res.title, res.visual) for res in r.results
               if res.visual is not None and (not only_flagged or res.level >= Level.WARN)]
    fig, axes = plt.subplots(1, len(panels), figsize=(6 * len(panels), 5))
    for ax, (title, img) in zip(np.atleast_1d(axes), panels):
        ax.imshow(img.convert("RGB"))
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def inspect(keyword):
    """파일 이름 일부로 사진 하나를 골라 모든 검사 그림을 봅니다. 예: inspect("IMG_0012")"""
    for r in reports:
        if keyword.lower() in r.item.name.lower():
            show_report(r, only_flagged=False)


min_level = LEVEL_BY_LABEL[SHOW_LEVEL]
flagged = sorted((r for r in reports if r.level >= min_level and not r.error), key=lambda r: -r.level)
print(f"'{SHOW_LEVEL}' 이상: {len(flagged)}장\n")
for r in flagged:
    show_report(r)

특정 사진을 모든 검사 그림과 함께 보려면 아래처럼 파일 이름 일부를 넣어 실행하세요.

In [ ]:
# inspect("IMG_0012")

## 7. 리포트 저장
HTML 리포트(브라우저로 열기)와 CSV 결과표(엑셀로 열기)를 `D:\data`에 저장합니다.

In [ ]:
order = sorted(reports, key=lambda r: -r.level)
write_html(order, [str(DATA_DIR)], REPORT_PATH)
df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")   # 엑셀에서 한글 안 깨지게
print("HTML 리포트:", REPORT_PATH)
print("CSV 결과표:", CSV_PATH)